# Kaggle avatar prototype
This notebook clones the public repository `koredeycode/kaggle-avatar-prototype` into Kaggle, starts the local FastAPI/WebSocket prototype in mock mode by default, and then exposes the browser path through Cloudflare Quick Tunnel. Public cloning needs no GitHub credential. A private clone is optional through a Kaggle Secret.

In [ ]:
import os
import secrets
import subprocess
import sys
import time
from pathlib import Path

GITHUB_REPO = os.getenv('GITHUB_REPO', 'https://github.com/koredeycode/kaggle-avatar-prototype.git').strip()
GITHUB_REF = os.getenv('GITHUB_REF', 'main').strip()
GITHUB_PRIVATE = os.getenv('GITHUB_PRIVATE', 'false').strip().lower() in {'1', 'true', 'yes'}
GITHUB_TOKEN_SECRET = os.getenv('GITHUB_TOKEN_SECRET', 'GITHUB_TOKEN').strip()
PROJECT_DIR = Path('/kaggle/working/kaggle-avatar-prototype')
if not PROJECT_DIR.exists():
    clone_env = os.environ.copy()
    if GITHUB_PRIVATE:
        from kaggle_secrets import UserSecretsClient
        github_token = UserSecretsClient().get_secret(GITHUB_TOKEN_SECRET)
        askpass = Path('/tmp/avatar-github-askpass.sh')
        askpass.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) printf \'%s\\n\' \'x-access-token\' ;;\n  *) printf \'%s\\n\' "$GITHUB_TOKEN" ;;\nesac\n')
        askpass.chmod(0o700)
        clone_env['GIT_ASKPASS'] = str(askpass)
        clone_env['GIT_TERMINAL_PROMPT'] = '0'
        clone_env['GITHUB_TOKEN'] = github_token
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', GITHUB_REF, GITHUB_REPO, str(PROJECT_DIR)], env=clone_env, check=True)
    clone_env.pop('GITHUB_TOKEN', None)
    if GITHUB_PRIVATE:
        askpass.unlink(missing_ok=True)
else:
    print('Using existing checkout:', PROJECT_DIR)
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR / 'src'))
runtime_token = secrets.token_urlsafe(32)
env = os.environ.copy()
env.update({
    'APP_HOST': '127.0.0.1',
    'APP_PORT': '8000',
    'RUNTIME_TOKEN': runtime_token,
    'MODEL_MODE': 'mock',
    'TTS_MODE': 'mock',
    'VAD_MODE': 'mock',
    'PUBLIC_MODE': 'true',
})
print('Runtime token:', runtime_token)
print('Keep this token private.')

In [ ]:
%pip install -e '.[dev,local-models]'
import json
import shutil
subprocess.run(['apt-get', 'update'], check=True)
subprocess.run(['apt-get', 'install', '-y', 'espeak-ng', 'libsndfile1', 'ffmpeg'], check=True)
subprocess.run([sys.executable, 'scripts/install_ollama.py'], check=True)
MODEL_ROOT = Path('/kaggle/working/avatar-models')
env['MODEL_ROOT'] = str(MODEL_ROOT)
env['HF_HOME'] = str(MODEL_ROOT / 'huggingface')
ollama_binary = shutil.which('ollama')
assert ollama_binary is not None
ollama_process = subprocess.Popen([ollama_binary, 'serve'], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(60):
    probe = subprocess.run([ollama_binary, 'list'], env=env, text=True, capture_output=True)
    if probe.returncode == 0:
        break
    time.sleep(1)
assert probe.returncode == 0, probe.stderr
subprocess.run([ollama_binary, 'pull', os.getenv('OLLAMA_MODEL', 'qwen3:8b')], env=env, check=True)
bootstrap = subprocess.run([sys.executable, 'scripts/bootstrap_models.py', '--root', str(MODEL_ROOT), '--sherpa', '--smart-turn', '--kokoro'], env=env, text=True, capture_output=True)
print(bootstrap.stdout)
print(bootstrap.stderr)
model_manifest = json.loads((MODEL_ROOT / 'model-manifest.json').read_text()) if (MODEL_ROOT / 'model-manifest.json').exists() else {}
model_ok = bootstrap.returncode == 0 and bool(model_manifest.get('sherpa_dir')) and bool(model_manifest.get('smart_turn'))
if model_ok:
    sherpa_dir = Path(model_manifest['sherpa_dir'])
    env['SHERPA_ENCODER'] = str(sherpa_dir / 'encoder-epoch-99-avg-1-chunk-16-left-128.int8.onnx')
    env['SHERPA_DECODER'] = str(sherpa_dir / 'decoder-epoch-99-avg-1-chunk-16-left-128.onnx')
    env['SHERPA_JOINER'] = str(sherpa_dir / 'joiner-epoch-99-avg-1-chunk-16-left-128.int8.onnx')
    env['SHERPA_TOKENS'] = str(sherpa_dir / 'tokens.txt')
    env['SMART_TURN_MODEL'] = model_manifest['smart_turn']
    env['MODEL_MODE'] = 'local'
    env['TTS_MODE'] = 'kokoro'
    env['VAD_MODE'] = 'silero'
    env['SMART_TURN_MODE'] = 'smart'
else:
    env['MODEL_MODE'] = 'mock'
    env['TTS_MODE'] = 'mock'
    env['VAD_MODE'] = 'mock'
    env['SMART_TURN_MODE'] = 'manual'
print('Local model profile:', env['MODEL_MODE'])

In [ ]:
preflight = subprocess.run([sys.executable, 'scripts/preflight.py'], env=env, text=True, capture_output=True, check=False)
print(preflight.stdout)
print(preflight.stderr)
assert preflight.returncode == 0

In [ ]:
app_log = open('/tmp/avatar-prototype-app.log', 'w', encoding='utf-8')
app_process = subprocess.Popen([sys.executable, '-m', 'avatar_prototype.main'], env=env, stdout=app_log, stderr=subprocess.STDOUT, cwd=PROJECT_DIR)
time.sleep(3)
assert app_process.poll() is None, 'avatar app exited early'
print('App PID:', app_process.pid)
print('Local URL: http://127.0.0.1:8000')

In [ ]:
import httpx
health = httpx.get('http://127.0.0.1:8000/healthz', timeout=10)
print(health.status_code, health.json())
health.raise_for_status()

## Cloudflare browser ingress
Cloudflare Quick Tunnel is required for the browser path. It is public, temporary, and not production hosting. The tunnel exposes only FastAPI and must be stopped before the session ends.

In [ ]:
cloudflared_setup = subprocess.run([sys.executable, 'scripts/install_cloudflared.py'], env=env, text=True, capture_output=True, check=False)
print(cloudflared_setup.stdout)
print(cloudflared_setup.stderr)
assert cloudflared_setup.returncode == 0
tunnel_log = open('/tmp/avatar-prototype-tunnel.log', 'w', encoding='utf-8')
tunnel_process = subprocess.Popen(['/usr/local/bin/cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:8000'], stdout=tunnel_log, stderr=subprocess.STDOUT, cwd=PROJECT_DIR)
time.sleep(5)
assert tunnel_process.poll() is None, 'cloudflared exited early'
print('Tunnel PID:', tunnel_process.pid)
print('Read the public URL from /tmp/avatar-prototype-tunnel.log')

In [ ]:
def cleanup():
    for process in (tunnel_process, app_process, ollama_process):
        if process is not None and process.poll() is None:
            process.terminate()
    for process in (tunnel_process, app_process, ollama_process):
        if process is None:
            continue
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            process.kill()
    app_log.close()
    tunnel_log.close()
cleanup()